# Imma Hungry (CIS 1921 Project)

Harry Li, Nabo Yu

We want to build an optimizer for the [Share Food Program](https://www.sharefoodprogram.org/philly-food-rescue).

In brief, the problem is as follows. We have a network of volunteer drivers to pick up perishable food from grocers, restaurants, and caterers, and they deliver them to affordable housing communities and nonprofits.

### Data Source

We use real world data from [Datasets - OpenDataPhilly](https://opendataphilly.org/datasets/?category=food), including [OpenMaps - OpenDataPhilly](https://opendataphilly.org/datasets/openmaps/) and [Free Food & Meal Sites - OpenDataPhilly](https://opendataphilly.org/datasets/free-meals/), and Google Maps API, including the [Compute Route Matrix](https://developers.google.com/maps/documentation/routes/compute-route-matrix-over) API.

- The OpenMaps dataset provides the geographic information of Philly.
- The Free Food & Meal Sites dataset provides the food donations information in Philly.
- The Compute Route Matrix API provides the travel route matrix (i.e., generalized distance matrix taking route and traffic into consideration) between the locations.

### Formulation

Let there be $X$ volunteer drivers with carrying capacity $c$ scattered around Philadelphia, $Y$ active food donors with $s$ hours of spoilage windows, and $Z$ drop-off locations with pantry capacity $p$. We will optimize the driver-donation assignments and the route for each assignment by minimizing *food waste* and maximizing *nutrient availability* in the process.

### Evaluation

We evaluate the system by running simulations of sampled location subsets in the OpenDataPhilly dataset.

1. With small-scale driver and location samples to verify our pipeline works.
2. With larger-scale driver and location samples to compare our solver with a baseline greedy strategy, to see if our solver does indeed improve upon naive approaches.
3. With larger-scale driver and location samples to compare our solver with some heuristic strategy, to see if our solver does indeed improve upon naive approaches.

In [21]:
import polars as pl

print("OK!")


OK!


## Data Source

In [22]:
def fetch_data(remote: str, local: str):
    import os
    if os.path.exists(local):
        return
    os.makedirs(os.path.dirname(local), exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(remote, local)

There is no public information on food donors. We use the information of restaurants instead, assuming they are willing to donate food.

In [23]:
# https://opendataphilly.org/datasets/licenses-and-inspections-business-licenses/
fetch_data(
    "https://phl.carto.com/api/v2/sql?q=SELECT+*,+ST_Y(the_geom)+AS+lat,+ST_X(the_geom)+AS+lng+FROM+business_licenses&filename=business_licenses&format=csv&skipfields=cartodb_id",
    "data/business_licenses.csv"
)

donor_locations = pl.read_csv(
	"data/business_licenses.csv",
	infer_schema_length=65536
)

donor_locations = donor_locations.filter(
	pl.col("licensestatus") == "Active",
	pl.col("licensetype").is_in(["Food Preparing and Serving",
								 "Food Preparing and Serving (30+)",
								 "Food Establishment, Retail Permanent"])
).select(
	pl.col("business_name"),
	pl.col("address"),
)
donor_locations = donor_locations.sample(32, seed=1921)
donor_locations

business_name,address
str,str
"""SUNNY SCOOP""","""1812 E PASSYUNK AVE"""
"""Al Tibyan Academy""","""139 N 63RD ST"""
"""Cheryl Haegele (Haegele's Bake…","""4164 BARNETT ST"""
"""AMERICAN BREAD CO LLC (AGENT S…","""1200 ARCH ST"""
"""Darman Learning Center LLC""","""7722 RICHARD ST"""
…,…
"""K S GROCERY LLC (KEISHA MINI …","""5301 OXFORD AVE"""
"""FORTUNE CHINESE RESTAURANT INC""","""1828 SOUTH ST"""
"""SOO LEE (Seaforest Bakeshop LL…","""625 S 16TH ST"""


We use the information of existing share food program sites.

In [24]:
# https://opendataphilly.org/datasets/free-meals/
fetch_data(
    "https://hub.arcgis.com/api/v3/datasets/5825a32bb8844bb097f7a16d4fbf4f23_0/downloads/data?format=csv&spatialRefId=3857&where=1%3D1",
    "data/free_meal_sites.csv"
)

dropoff_locations = pl.read_csv(
    "data/free_meal_sites.csv"
)

dropoff_locations = dropoff_locations.filter(
    pl.col("category").str.contains("SHARE FOOD PROGRAM")
).select(
    pl.col("site_name"),
    pl.col("address"),
)
dropoff_locations = dropoff_locations.sample(32, seed=1921)
dropoff_locations

site_name,address
str,str
"""Hickman Temple Daycare""","""1220 S. 58th St"""
"""Helping Hands Nonprofit Organi…","""4667 Paul St."""
"""Reconciliation And Liberty Bib…","""6027 Chestnut St"""
"""Mount Hebron Baptist Church""","""1419 Wharton St"""
"""St Paul A.M.E. Church""","""8398 Lindbergh Blvd"""
…,…
"""Mantua Haverford Community Cen…","""769 N 39th St"""
"""OCCCDA (Howell St.)""","""900 E Howell St"""
"""Church of Pentecost""","""2530 Wharton St"""


We use Routes API (Google Maps API) to compute the distance matrix from donors locations + drop-off locations to drop-off locations (because a driver is likely not to drive back to donors).

In [25]:
def fetch_matrix():
    import os
    from google.api_core.client_options import ClientOptions
    import google.maps.routing_v2 as routes

    CACHE_PATH = "data/route_matrix.csv"
    if os.path.exists(CACHE_PATH):
        return pl.read_csv(CACHE_PATH)

    # The API key is redacted for security.
    client = routes.RoutesClient(
        client_options=ClientOptions(
            api_key="???"
        )
    )

    # origins: donors + drop-offs
    origins = (
        donor_locations.select(pl.col("address")).to_series().to_list() +
        dropoff_locations.select(pl.col("address")).to_series().to_list()
    )
    # destinations: drop-offs only
    destinations = (
        dropoff_locations.select(pl.col("address")).to_series().to_list()
    )
    request_origins = [
        routes.RouteMatrixOrigin(waypoint=routes.Waypoint(
            address=f"{address}, Philadelphia, PA"
        ))
        for address in origins
    ]
    request_destinations = [
        routes.RouteMatrixDestination(waypoint=routes.Waypoint(
            address=f"{address}, Philadelphia, PA"
        ))
        for address in destinations
    ]

    # API limit: total addresses (origins + destinations) per request <= 50
    # => batch_size <= 50 - len(destinations)
    batch_size = 50 - len(destinations)

    rows = []
    for i in range(0, len(origins), batch_size):
        batch_origins = request_origins[i: i + batch_size]
        response = client.compute_route_matrix(
            routes.ComputeRouteMatrixRequest(
                origins=batch_origins,
                destinations=request_destinations,
            ),
            metadata=[("x-goog-fieldmask",
                       "originIndex,destinationIndex,duration,distanceMeters,status")],
        )
        for element in response:
            rows.append({
                "origin": origins[i + element.origin_index],
                "destination": destinations[element.destination_index],
                "distance": element.distance_meters,
                "duration": element.duration.seconds,
            })

    df = pl.DataFrame(rows)
    df.write_csv(CACHE_PATH)
    return df

# distance (in meters)
# duration (in seconds)
route_matrix = fetch_matrix()
route_matrix = route_matrix.sort(["origin", "destination"])
route_matrix

origin,destination,distance,duration
str,str,i64,i64
"""100 E. Lehigh Ave.""","""100 E. Lehigh Ave.""",0,0
"""100 E. Lehigh Ave.""","""1220 S. 58th St""",16839,1713
"""100 E. Lehigh Ave.""","""1419 Wharton St""",10607,1362
"""100 E. Lehigh Ave.""","""1429 W Clearfield St""",3359,628
"""100 E. Lehigh Ave.""","""1510 W Oxford St""",4657,870
…,…,…,…
"""9745R E ROOSEVELT BLVD""","""735 W. Butler St.""",16278,1720
"""9745R E ROOSEVELT BLVD""","""769 N 39th St""",29157,2082
"""9745R E ROOSEVELT BLVD""","""8398 Lindbergh Blvd""",39266,2228
